In [1]:
# Cella 1: Setup
import sys
from pathlib import Path
import torch
from transformers import TrainingArguments, Trainer

ROOT = Path.cwd().resolve().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from project_paths import get_paths
from teacher_finetune_headtail import (
    load_flat_dataset, TeacherModelConfig, build_teacher_model,
    build_teacher_tokenizer, build_collator, compute_metrics,
    get_llrd_optimizer_parameters, bf16_supported
)

paths = get_paths(ROOT)

c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_ds = load_flat_dataset(paths.data_processed / "train_120k_ht.parquet")
val_ds = load_flat_dataset(paths.data_processed / "val_120k_ht.parquet")

MODEL_NAME = "bert-large-uncased"
tokenizer = build_teacher_tokenizer(MODEL_NAME)
collator = build_collator(tokenizer)

model_cfg = TeacherModelConfig(
    model_name = MODEL_NAME,
    gradient_checkpointing=True,
    hidden_dropout_prob = 0.1
)

model = build_teacher_model(model_cfg)

if model_cfg.gradient_checkpointing:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-large-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
LR_MAX = 2e-5
DECAY_RATE = 0.95
WEIGHT_DECAY = 0.01

optimizer_grouped_parameters = get_llrd_optimizer_parameters(
    model,
    learning_rate = LR_MAX,
    weight_decay = WEIGHT_DECAY,
    layer_decay = DECAY_RATE
)

optimizer = torch.optim.AdamW(optimizer_grouped_parameters)

args = TrainingArguments(
    output_dir=str(paths.checkpoints / "teacher_bert_large_llrd"),
    per_device_train_batch_size=4,   
    gradient_accumulation_steps=8,   
    num_train_epochs=2,              
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    bf16=bf16_supported(),
    fp16=not bf16_supported(),
    report_to="none"
)

class CustomTrainer(Trainer):
    def __init__(self, *args, pos_weight_value=None, **kwargs):
        super().__init__(*args, **kwargs)
        # Salviamo il valore del peso (es. 2.8)
        self.pos_weight_value = pos_weight_value

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 1. Rimuoviamo labels dall'input per evitare calcoli automatici errati
        labels = inputs.pop("labels")
        
        # 2. Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # 3. Fix Dimensioni [Batch, 1] -> [Batch]
        if logits.shape != labels.shape:
            logits = logits.view(-1)
            labels = labels.view(-1)
            
        # 4. Configurazione Loss con Peso
        if self.pos_weight_value is not None:
            # Creiamo il tensore peso sullo stesso device (GPU) dei logits
            weight_tensor = torch.tensor([self.pos_weight_value], device=logits.device)
            loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=weight_tensor)
        else:
            loss_fct = torch.nn.BCEWithLogitsLoss()
            
        loss = loss_fct(logits, labels.float())
        
        return (loss, outputs) if return_outputs else loss


trainer = CustomTrainer(  
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),
    pos_weight_value=2.80
)

print("🚀 Starting LLRD Training (con Shape Fix)...")
trainer.train()

final_path = paths.checkpoints / "teacher_bert_large_final_headtail"
trainer.save_model(str(final_path))
print(f"Model saved to {final_path}")

🚀 Starting LLRD Training (con Shape Fix)...


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Auc
250,0.905500,0.891532,0.646000,0.537960,0.409573,0.783587,0.749004
500,0.894800,0.854184,0.700833,0.555693,0.455930,0.711343,0.776459
750,0.880600,0.914335,0.759750,0.515055,0.548942,0.485108,0.776905
1000,0.850900,0.810896,0.748000,0.570210,0.517010,0.635615,0.795381
1250,0.807100,0.796220,0.728000,0.581646,0.488377,0.718948,0.802555
1500,0.828400,0.825597,0.774167,0.576165,0.568870,0.583650,0.803664
1750,0.832000,0.801069,0.695750,0.570824,0.453747,0.769328,0.803783
2000,0.860100,0.793116,0.745500,0.587409,0.512011,0.688847,0.806393
2250,0.823800,0.785153,0.732583,0.584057,0.494187,0.713878,0.807131
2500,0.829300,0.823589,0.782250,0.578072,0.589397,0.567174,0.804179


Model saved to C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\checkpoints\teacher_bert_large_final_headtail
